# 10. Methodology (Planned for Phase 2)

## Forecasting Leptospirosis Cases from Climate Data in a Philippine City

This section explains **how the dataset will be prepared, modeled, and evaluated** in Phase 2.

The main goal is to answer:

> **Can historical climate conditions help forecast the number of weekly leptospirosis cases in Iloilo City?**

This is a **forecasting problem** because the model will use information from earlier weeks to predict case counts in later weeks.

---

### Plain-language overview

The project will:

1. collect the relevant CCHAIN tables;
2. focus on **Iloilo City** and the common study period;
3. convert the data into **weekly observations**;
4. combine climate and leptospirosis records;
5. check the data for missing or suspicious values;
6. create useful forecasting features such as **previous-week rainfall**;
7. split the data by time rather than randomly;
8. train several models;
9. evaluate which model predicts future case counts best; and
10. interpret which climate variables appear most useful for forecasting.

The methodology is designed to prevent the model from accidentally learning from the future.

## 10.1 Modeling Objective

The objective of Phase 2 is to determine whether historical climate conditions can be used to forecast **weekly leptospirosis case counts in Iloilo City**.

The unit of analysis will be:

> **One city-week**

For example, one row in the final modeling dataset may represent:

| Week | Leptospirosis cases | Rainfall | Temperature | Humidity |
|---|---:|---:|---:|---:|
| 2008 Week 1 | 2 | 41.2 mm | 27.1 °C | 80.4% |
| 2008 Week 2 | 4 | 85.6 mm | 26.8 °C | 83.1% |

The dependent variable, or **target**, is:

\[
y_t = \text{number of reported leptospirosis cases during week } t
\]

The predictors will be based on the climate and atmospheric variables available in the CCHAIN dataset.

These may include variables such as:

- rainfall or precipitation;
- temperature;
- humidity;
- other available atmospheric measurements; and
- time-based features such as seasonality.

Only variables that are actually available and sufficiently complete in the dataset will be retained.

## 10.2 Overall Pipeline

The planned workflow is:

```text
Raw CCHAIN tables
        ↓
Select Iloilo City and the study period
        ↓
Create a complete weekly timeline
        ↓
Aggregate cases and climate variables by week
        ↓
Merge climate and leptospirosis data
        ↓
Validate records and inspect missing/extreme values
        ↓
Create forecasting features
(lags, rolling summaries, seasonality)
        ↓
Chronological train / validation / test split
        ↓
Fit preprocessing using training data only
        ↓
Train baseline and candidate models
        ↓
Tune using validation data / time-series cross-validation
        ↓
Final evaluation on the untouched test set
        ↓
Interpret results
```

### Why this order matters

The order is important because some preprocessing steps can accidentally reveal information from later years.

For example, if the mean temperature of the **entire dataset** were used to fill missing values before splitting the data, information from the future test period would indirectly enter the training process.

This is called **data leakage**.

To avoid this, any preprocessing step that learns values from the data will be fitted using the **training set only**.

## 10.3 Data Preparation and Preprocessing

The raw CCHAIN tables may have different formats, time resolutions, missing observations, and overlapping variables.

The following preprocessing steps are planned.

| Issue | Planned technique | Reason |
|---|---|---|
| Different time resolutions | Aggregate all variables to weekly observations | Climate and disease data need the same time unit before they can be modeled together |
| Missing weeks | Create a continuous weekly calendar | Prevents weeks from disappearing simply because no row exists |
| Missing climate observations | Inspect the amount and pattern of missingness; use limited forward-fill or training-derived imputation where appropriate | Missing values should be handled without using future information |
| Extreme values | Investigate rather than automatically remove | Extremely high rainfall or case counts may represent real events rather than bad data |
| Skewed case counts | Keep the original count target first; optionally compare a `log1p` version for linear regression | Transformation should only be used if model diagnostics show it is useful |
| Delayed climate effects | Create lagged climate variables | Weather conditions may affect reported cases several weeks later |
| Accumulated climate exposure | Create backward-looking rolling summaries | Several wet weeks may matter more than only one week's rainfall |
| Seasonality | Use cyclical week-of-year features | Leptospirosis and climate conditions may follow annual patterns |
| Multicollinearity | Inspect correlation and VIF for linear models; remove redundant predictors when justified | Highly correlated variables can make linear-model coefficients unstable |
| Feature scaling | Apply only to models that need it, fitted on training data only | Prevents leakage and avoids unnecessary scaling for tree-based models |

### Important note about zero-case weeks

A missing row is **not automatically the same as zero cases**.

A week should only be assigned a leptospirosis case count of zero if the source data confirms that the missing record truly represents **no reported cases**, rather than unavailable or incomplete surveillance data.

This distinction matters because incorrectly replacing missing disease records with zero can bias the model.

## 10.4 Feature Engineering

Feature engineering means creating new variables that may help the model make better predictions.

The main types of features planned for this project are:

1. **lagged climate variables**;
2. **rolling climate summaries**; and
3. **seasonality features**.

### A. Lagged climate variables

A lagged variable uses the value from an earlier week.

For example:

- `rainfall_t-1` = rainfall one week earlier;
- `rainfall_t-2` = rainfall two weeks earlier;
- `rainfall_t-4` = rainfall four weeks earlier.

Candidate lags may initially cover approximately **1 to 8 previous weeks**.

For rainfall:

\[
Rainfall_{t-1}, Rainfall_{t-2}, \ldots, Rainfall_{t-8}
\]

Similar lagged variables may be created for temperature and other retained climate features.

#### Why use lags?

Leptospirosis cases may not appear during the exact same week as a heavy rainfall event.

A person may be exposed during flooding, develop symptoms later, seek medical attention later, and only then be recorded as a case.

Therefore, previous climate conditions may be more useful than same-week climate alone.

### B. Rolling summaries

A rolling summary combines several earlier weeks into one value.

For example, a four-week average rainfall feature may be:

\[
RainfallMean_{t,4}
=
\frac{Rainfall_{t-1}+Rainfall_{t-2}+Rainfall_{t-3}+Rainfall_{t-4}}{4}
\]

Possible windows include:

- 2 weeks;
- 4 weeks; and
- 8 weeks.

These features can represent **accumulated exposure** to prolonged wet conditions.

### C. Seasonality

Climate and disease patterns may repeat at similar times each year.

Instead of treating the week number as a simple number from 1 to 52, the project may use cyclical features:

\[
sin\left(\frac{2\pi \cdot week}{52}\right)
\]

\[
cos\left(\frac{2\pi \cdot week}{52}\right)
\]

This helps the model understand that:

> **Week 52 and Week 1 are next to each other in the yearly cycle.**

A normal numeric week variable would incorrectly make them appear very far apart.

### Preventing leakage during feature engineering

All forecasting features must be **causal**.

That means a prediction for week \(t\) may only use information from week \(t-1\) or earlier.

**Allowed:** predict Week 10 using Week 9, Week 8, Week 7, and earlier data.

**Not allowed:** predict Week 10 using Week 11 or Week 12 data.

Future observations must never be used to construct predictors.

## 10.5 Data Split

Because the observations form a **time series**, the dataset will **not be randomly shuffled**.

A normal random split would mix earlier and later years together. That would make the evaluation unrealistic because the model could learn from observations that happened after the dates it is supposedly predicting.

Instead, the project will use a **chronological split**.

### Proposed split

| Set | Proposed period | Purpose |
|---|---|---|
| Train | 2008–2018 | Fit the model |
| Validation | 2019–2020 | Select features and tune model settings |
| Test | 2021–2022 | Final unbiased evaluation |

The exact boundaries may be adjusted slightly to preserve complete epidemiological weeks or depending on final data availability.

However, the test set should remain the **latest contiguous period**.

```text
2008                                                2022
|---------------- TRAIN ----------------|-- VALID --|-- TEST --|
Past ----------------------------------------------------> Future
```

### Why have three sets?

**Training set** — used to teach the model.

**Validation set** — used to compare model settings and choose the best configuration.

**Test set** — used only at the end to estimate how well the final model performs on truly unseen future data.

The test set must not be used to decide:

- which features to keep;
- which lag window works best;
- which hyperparameters to use;
- which model is selected; or
- how missing values should be handled.

## 10.6 Optional Time-Series Cross-Validation

If more reliable model tuning is needed, the project may use **expanding-window time-series cross-validation** within the training period.

Unlike ordinary cross-validation, the order of time is preserved.

```text
Fold 1:  TRAIN --------> VALIDATE
Fold 2:  TRAIN ----------------> VALIDATE
Fold 3:  TRAIN ------------------------> VALIDATE

Past ----------------------------------------------> Future
```

Each later fold gives the model more historical training data while always validating on observations that occur afterward.

This better reflects the real forecasting task.

## 10.7 Forecasting Models

Because the outcome is the **number of weekly leptospirosis cases**, the task is primarily a **regression/count forecasting problem**.

The project will compare simple baselines against more advanced models.

| Model | Purpose |
|---|---|
| **Naive baseline** | Predicts using the most recent observed case count; provides a basic benchmark |
| **Seasonal naive baseline** | Predicts using the corresponding week from the previous year; provides a seasonal benchmark |
| **Multiple Linear Regression** | Simple and interpretable relationship between climate predictors and weekly case counts |
| **Poisson Regression** | Natural starting model for a non-negative count outcome |
| **Negative Binomial Regression** | Useful if the variance in case counts is much larger than the mean |
| **Random Forest Regressor** | Captures nonlinear relationships and interactions |
| **Gradient Boosting Regressor** | Strong candidate for structured/tabular data and nonlinear effects |

### Why not Logistic Regression?

Logistic Regression is mainly used when the target has categories such as:

- outbreak / no outbreak;
- disease / no disease;
- yes / no.

That is **not the current target**.

The current target is a count:

\[
0,\ 1,\ 2,\ 3,\ 4,\ldots
\]

Therefore, this project is currently framed as a **count forecasting/regression task**, not a classification task.

Classification should only be added if the research question is explicitly changed to something like:

> **Will next week be an outbreak week: yes or no?**

That would be a separate modeling objective.

## 10.8 Model Complexity and Overfitting

The dataset contains roughly **780 weekly observations** across the 2008–2022 period.

That is enough for meaningful modeling, but it is still relatively small compared with many machine-learning datasets.

Therefore, the project should avoid creating an unnecessarily large number of features or highly complex models.

Too much model complexity can cause **overfitting**.

Overfitting happens when a model learns the training data extremely well but performs poorly on future unseen data.

To reduce this risk:

- use a limited and justified set of climate features;
- keep lag windows manageable;
- tune models using validation data rather than the test set;
- compare complex models against simple baselines;
- prefer the simpler model if two models perform similarly; and
- evaluate performance on later unseen years.

## 10.9 Model Tuning

Model tuning means selecting model settings that work best on validation data.

Possible tuning decisions include:

| Component | Example values |
|---|---|
| Climate lag window | 1, 2, 4, 6, or 8 weeks |
| Rolling window | 2, 4, or 8 weeks |
| Random Forest | number of trees, maximum depth, minimum leaf size |
| Gradient Boosting | number of estimators, learning rate, tree depth |

The best settings will be chosen using:

- the validation set; or
- time-series cross-validation within the training period.

The final test set will **not** be used for tuning.

## 10.10 Evaluation Metrics

Because the target is the **weekly number of leptospirosis cases**, the project will use regression/count forecasting metrics.

The **primary metric will be Mean Absolute Error (MAE)**.

### Mean Absolute Error

\[
MAE =
\frac{1}{n}
\sum_{i=1}^{n}
|y_i-\hat{y}_i|
\]

MAE measures the average absolute difference between the actual and predicted case counts.

For example:

> If MAE = 2.3, the model is wrong by about **2.3 leptospirosis cases per week on average**.

This makes MAE easy to explain to both technical and non-technical audiences.

### Secondary metrics

| Metric | Meaning |
|---|---|
| **MAE** | Average absolute forecasting error in weekly cases |
| **RMSE** | Similar to MAE, but gives a larger penalty to large mistakes |
| **R²** | Measures how much of the variation in weekly case counts is explained by the model |
| **Poisson or count deviance** | May be reported for count-regression models |

All metrics should be reported on the **original case-count scale** whenever possible so the errors remain easy to interpret.

### Why not use MAPE as the main metric?

MAPE calculates percentage error.

However, the dataset may contain weeks with:

\[
Actual\ Cases = 0
\]

Percentage errors become undefined or misleading when the actual value is zero.

For this reason, **MAE is more suitable as the main metric**.

## 10.11 Model Interpretation

The goal is not only to find the model with the lowest error.

The project will also investigate **what the model learned**.

The analysis will examine:

- which climate variables are most useful for forecasting;
- which lag periods provide the strongest predictive information;
- whether rolling climate conditions improve prediction;
- whether seasonality improves forecasting performance;
- whether nonlinear models perform meaningfully better than simple models; and
- whether the selected model beats naive and seasonal-naive baselines.

For **Linear, Poisson, and Negative Binomial Regression**, coefficient estimates may be examined.

For **Random Forest and Gradient Boosting**, feature importance or another suitable model-interpretation method may be used.

### Important limitation: prediction is not causation

The methodology is designed to evaluate **predictive usefulness**.

If rainfall is useful for predicting future leptospirosis cases, that does not automatically prove that rainfall directly caused the observed increase in cases.

The model may identify associations that are useful for forecasting without establishing a causal relationship.

Therefore, conclusions should use terms such as:

- **associated with**;
- **predictive of**;
- **useful for forecasting**; and
- **related to**.

Avoid making causal claims unless the study design specifically supports them.

## 10.12 Final Methodology Summary

The final Phase 2 methodology can be summarized as:

```text
CCHAIN data
    ↓
Iloilo City + study period
    ↓
Weekly climate and case dataset
    ↓
Data-quality checks
    ↓
Lagged climate + rolling + seasonal features
    ↓
Chronological train / validation / test split
    ↓
Baseline models
    ↓
Regression and count models
    ↓
Tree-based models
    ↓
Validation and tuning
    ↓
Final test-set evaluation
    ↓
Interpretation of climate predictors and forecasting performance
```

### In one sentence

> **Historical climate observations will be transformed into weekly, backward-looking forecasting features and used to predict future leptospirosis case counts, with models evaluated on later unseen years to simulate a real forecasting scenario.**

## 10.13 Short Version for the Presentation Slide

For the actual presentation, the methodology can be simplified to six stages:

### 01 — Data Selection
**CCHAIN → Iloilo City → 2008–2022**

### 02 — Preprocessing
**Weekly aggregation → merge → data-quality checks**

### 03 — Feature Engineering
**Climate lags → rolling summaries → seasonality**

### 04 — Temporal Split
**Train → Validation → Test**

### 05 — Modeling
**Baselines → Regression / Count Models → Tree-Based Models**

### 06 — Evaluation
**MAE → RMSE → R² → Interpretation**

A useful line to explain the methodology verbally is:

> **Because this is a forecasting problem, the data will be split chronologically rather than randomly so that the model is always evaluated on future observations it has not seen before.**

## 10.14 Methodological Decisions to Confirm After EDA

Some choices should **not be finalized before the exploratory analysis is complete**.

The team should confirm the following after examining the actual dataset:

- the exact available climate variables;
- the amount and pattern of missing climate data;
- whether missing case rows truly mean zero cases;
- whether weekly case counts are strongly skewed;
- whether case counts show overdispersion;
- which climate variables are highly correlated;
- the most reasonable lag range;
- whether there is strong annual seasonality;
- whether extreme observations are valid events or data errors; and
- whether the proposed 2008–2018 / 2019–2020 / 2021–2022 split is appropriate after accounting for data availability.

These decisions should be based on the actual EDA findings rather than assumptions made in advance.